#Carga de información ya corregida


In [ ]:
# ==============================================================================
# BLOQUE 7: ETL Y ACTUALIZACIÓN DEL DATASET MAESTRO (BASE ACUMULADA)
# ==============================================================================
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

display(Markdown("## 🏗️ Reconstrucción del Dataset Maestro (Formato Acumulado)"))

# 1. RUTAS DE ARCHIVOS
ruta_parquet_crudo = '/content/Consolidado_Suministros_Maestro.parquet'
ruta_excel = '/content/Bases_homologadas.xlsx'
ruta_2025 = '/content/2025_Primas_Fasecolda.xlsx'
ruta_2026 = '/content/2026_Primas_Fasecolda.xlsx'

# 2. CARGA Y UNIÓN
df_base = pd.read_parquet(ruta_parquet_crudo)
df_2025 = pd.read_excel(ruta_2025)
df_2026 = pd.read_excel(ruta_2026)
df = pd.concat([df_base, df_2025, df_2026], ignore_index=True)

df['FECHA'] = pd.to_datetime(df['FECHA'], format='%d/%m/%Y', errors='coerce')
df['AÑO'] = df['FECHA'].dt.year

df_homolog_ramos = pd.read_excel(ruta_excel, sheet_name='RAMO_FASECOLDA')
df_homolog_companias = pd.read_excel(ruta_excel, sheet_name='COMPANIA_FASECOLDA')
df_sucursal_regional = pd.read_excel(ruta_excel, sheet_name='SUCURSAL_REGIONAL')

# 3. HOMOLOGACIÓN
dict_ramos = dict(zip(df_homolog_ramos['RAMO'], df_homolog_ramos['HOMOLOG_RAMO']))
dict_companias = dict(zip(df_homolog_companias['COMPANIA'], df_homolog_companias['HOMOLOG_COMPANIA']))
dict_companias['ARL SURA'] = 'SURAMERICANA'
dict_companias['AXA COLPATRIA'] = 'AXA'
dict_companias['AXA COLPATRIA GENERALES'] = 'AXA'
dict_ramos['COLECTIVO Y GRUPO '] = 'GRUPO'

df['RAMOS'] = df['RAMOS'].apply(lambda x: dict_ramos.get(x, x))
df['COMPANIA'] = df['COMPANIA'].apply(lambda x: dict_companias.get(x, x))

df['CIUDAD'] = df['CIUDAD'].astype(str).str.strip().str.upper()
df_sucursal_regional['CIUDAD'] = df_sucursal_regional['CIUDAD'].astype(str).str.strip().str.upper()

df = df.merge(df_sucursal_regional[['CIUDAD', 'SUCURSAL', 'REGIONAL']], on='CIUDAD', how='left')
df['SUCURSAL'] = df['SUCURSAL'].fillna('SIN ASIGNAR')
df['REGIONAL'] = df['REGIONAL'].fillna('SIN ASIGNAR')

# 4. AGRUPACIÓN (MANTENIENDO EL ACUMULADO)
columnas_agrupacion = ['AÑO', 'FECHA', 'COMPANIA', 'RAMOS', 'CIUDAD', 'SUCURSAL', 'REGIONAL']
df_maestro = df.groupby(columnas_agrupacion)['TOTAL'].sum().reset_index()
df_maestro = df_maestro.sort_values(by=['AÑO', 'COMPANIA', 'RAMOS', 'CIUDAD', 'FECHA'])

# 5. GUARDADO
ruta_salida = '/content/Base_Maestra_Acumulada.parquet'
df_maestro.to_parquet(ruta_salida, index=False)

display(Markdown(f"### 💾 ¡Parquet Acumulado Guardado Exitosamente! (`{ruta_salida}`)"))

#Cargue de la información

In [ ]:
# ==============================================================================
# BLOQUE 0: CARGA DIRECTA DEL DATASET MAESTRO ACUMULADO
# ==============================================================================
# 1. Importar las librerías necesarias
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------
# PASO 1: DEFINIR LA RUTA DEL ARCHIVO
# ---------------------------------------------------------

# OPCIÓN A: Si subiste el archivo directamente a la sesión temporal de Colab (ícono de carpeta a la izquierda):
ruta_archivo = '/content/Base_Maestra_Acumulada.parquet'

# OPCIÓN B: Si el archivo está en tu Google Drive (Descomenta las siguientes 3 líneas si es tu caso):
# from google.colab import drive
# drive.mount('/content/drive')
# ruta_archivo = '/content/drive/MyDrive/TU_CARPETA_AQUI/Base_Maestra_Acumulada.parquet'

# ---------------------------------------------------------
# PASO 2: CARGAR LA BASE DE DATOS
# ---------------------------------------------------------

# Cargar el archivo parquet (Colab ya tiene el motor 'pyarrow' instalado por defecto)
df_maestro = pd.read_parquet(ruta_archivo)

# ---------------------------------------------------------
# PASO 3: INSPECCIÓN INICIAL (EDA Rápido)
# ---------------------------------------------------------

print("1. Información general de la base (Tipos de datos y nulos):")
df_maestro.info()

print("\n2. Resumen estadístico inicial de las variables numéricas:")
# Con display() la tabla se renderiza con el formato bonito e interactivo de Colab
display(df_maestro.describe().T)

print("\n3. Vista previa de las primeras 5 filas:")
display(df_maestro.head())

In [ ]:
# ==============================================================================
# BLOQUE 8.1: CREACIÓN DEL DATAFRAME DE CIERRE ANUAL (DICIEMBRE)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Filtramos estrictamente el mes 12 (la "foto final" del año)
df_anual = df_maestro[df_maestro['FECHA'].dt.month == 12].copy()

# Generar listas de opciones para los filtros de la interfaz
companias_unicas = sorted(df_anual['COMPANIA'].dropna().unique().tolist())
ramos_unicos = sorted(df_anual['RAMOS'].dropna().unique().tolist())
regionales_unicas = sorted(df_anual['REGIONAL'].dropna().unique().tolist())
ciudades_unicas = sorted(df_anual['CIUDAD'].dropna().unique().tolist())
sucursales_unicas = sorted(df_anual['SUCURSAL'].dropna().unique().tolist())
anios_disponibles = sorted(df_anual['AÑO'].dropna().unique().tolist())
ultimo_anio = anios_disponibles[-1] if anios_disponibles else None

print(f"✅ Se ha creado 'df_anual' con la foto de cierre de diciembre ({len(df_anual)} registros).")

# Anual por ramo

In [ ]:
# ==============================================================================
# BLOQUE 8.2: TABLERO YTD - RAMO (VERSIÓN DINÁMICA DE COMPAÑÍAS)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted(df_fc['AÑO'].dropna().unique().tolist())
ramos_unicos = sorted(df_fc['RAMOS'].dropna().unique().tolist())
companias_unicas = sorted(df_fc['COMPANIA'].dropna().unique().tolist())

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_num = df_fc[df_fc['AÑO'] == ultimo_anio]['FECHA'].dt.month.max()
ultimo_mes_txt = meses_dict.get(ultimo_mes_num, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_ramo = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_ramo = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_ramo = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_cias_ramo = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '120px'})
sel_ramos_ramo = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})

btn_ramo = widgets.Button(description='Generar Tablero Ramo', button_style='primary', icon='list', layout={'width': '300px', 'height': '40px'})

ui_ramo = widgets.VBox([
    widgets.HBox([dd_anio_ramo, dd_mes_ramo, sel_anios_atras_ramo]),
    widgets.HBox([sel_cias_ramo, sel_ramos_ramo]),
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Filtra por el acumulado YTD al mes seleccionado. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_ramo
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})

out_ramo = widgets.Output()

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_tablero_ramo(b):
    with out_ramo:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_ramo.value, list(sel_anios_atras_ramo.value)
        companias_obj, ramo_obj = list(sel_cias_ramo.value), list(sel_ramos_ramo.value)
        mes_str = dd_mes_ramo.value
        mes_num = meses_inversos[mes_str]

        if not companias_obj or not ramo_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones (Compañías, Ramos o Años Atrás)."))
            return

        display(Markdown(f"# 📊 Análisis YTD por Ramo (Acumulado a {mes_str})"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            df_base = df_fc[(df_fc['AÑO'].isin([anio_base, anio_ant])) &
                            (df_fc['FECHA'].dt.month == mes_num) &
                            (df_fc['RAMOS'].isin(ramo_obj))].copy()

            if df_base.empty:
                display(Markdown(f"⚠️ No hay datos para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            df_base['ETIQUETA_TIEMPO'] = np.where(df_base['AÑO'] == anio_base, lbl_actual, lbl_anterior)

            df_mkt = df_base.groupby(['ETIQUETA_TIEMPO', 'RAMOS'])['TOTAL'].sum().reset_index()
            df_mkt['COMPANIA'] = 'MERCADO'
            df_cias = df_base[df_base['COMPANIA'].isin(companias_obj)].groupby(['ETIQUETA_TIEMPO', 'COMPANIA', 'RAMOS'])['TOTAL'].sum().reset_index()

            df_pivot = pd.concat([df_cias, df_mkt]).pivot_table(index='RAMOS', columns=['COMPANIA', 'ETIQUETA_TIEMPO'], values='TOTAL', aggfunc='sum').fillna(0)
            if df_pivot.empty: continue

            df_pivot.loc['TOTAL'] = df_pivot.sum()

            frames, orden_col = [], ['MERCADO'] + companias_obj

            ayer_mkt = df_pivot.get(('MERCADO', lbl_anterior), pd.Series(0, index=df_pivot.index))
            hoy_mkt = df_pivot.get(('MERCADO', lbl_actual), pd.Series(0, index=df_pivot.index))
            var_mkt = pd.Series(np.nan, index=df_pivot.index)
            var_mkt[ayer_mkt != 0] = (hoy_mkt[ayer_mkt != 0] / ayer_mkt[ayer_mkt != 0]) - 1

            for cia in orden_col:
                if cia not in df_pivot.columns.get_level_values(0): continue
                ayer = df_pivot.get((cia, lbl_anterior), pd.Series(0, index=df_pivot.index))
                hoy = df_pivot.get((cia, lbl_actual), pd.Series(0, index=df_pivot.index))
                var = pd.Series(np.nan, index=df_pivot.index)
                var[ayer != 0] = (hoy[ayer != 0] / ayer[ayer != 0]) - 1

                if cia == 'MERCADO':
                    frames.append(pd.DataFrame({
                        (cia, lbl_anterior): ayer,
                        (cia, lbl_actual): hoy,
                        (cia, 'Var'): var
                    }))
                else:
                    veces = pd.Series(np.nan, index=df_pivot.index)
                    m_v = var_mkt.notna() & (var_mkt != 0)
                    veces[m_v] = var[m_v] / var_mkt[m_v].abs()
                    frames.append(pd.DataFrame({
                        (cia, lbl_anterior): ayer,
                        (cia, lbl_actual): hoy,
                        (cia, 'Var'): var,
                        (cia, 'Veces'): veces
                    }))

            if not frames: continue
            tablero = pd.concat(frames, axis=1)

            # Manejo de "Sin registros"
            for cia in orden_col:
                if (cia, lbl_anterior) in tablero.columns:
                    mask = (tablero[(cia, lbl_anterior)] == 0) & (tablero[(cia, lbl_actual)] == 0) & (tablero.index != 'TOTAL')
                    if mask.any():
                        cols_to_cast = [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')]
                        if (cia, 'Veces') in tablero.columns: cols_to_cast.append((cia, 'Veces'))
                        for col in cols_to_cast:
                            tablero[col] = tablero[col].astype(object)
                        tablero.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                        tablero.loc[mask, (cia, 'Var')] = "-"
                        if (cia, 'Veces') in tablero.columns: tablero.loc[mask, (cia, 'Veces')] = "-"

            orden_filas = sorted([r for r in ramo_obj if r in tablero.index]) + (['TOTAL'] if 'TOTAL' in tablero.index else [])
            tablero = tablero.loc[orden_filas]

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
            def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
            def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in tablero.columns}

            # ==================================================================
            # CORRECCIÓN DEFINITIVA DE ESTILOS (Soporte Pandas 2.0+)
            # ==================================================================
            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for cia in orden_col:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        df_st[(cia, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (cia, 'Veces') in data.columns:
                        v_veces = pd.to_numeric(data[(cia, 'Veces')], errors='coerce')
                        df_st[(cia, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                 'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                 'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_veces]

                    m_sin = data[(cia, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (cia, 'Veces') in data.columns: df_st.loc[m_sin, (cia, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(tablero.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

        gc.collect()

btn_ramo.on_click(generar_tablero_ramo)
display(ui_ramo, out_ramo)

# Por ciudad

In [ ]:
# ==============================================================================
# BLOQUE 8.4: TABLERO YTD - CIUDAD (VERSIÓN DINÁMICA DE COMPAÑÍAS)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# 1. CARGA DE DATOS
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# 2. PARÁMETROS PARA WIDGETS
anios_disponibles = sorted(df_fc['AÑO'].dropna().unique().tolist())
ramos_unicos = sorted(df_fc['RAMOS'].dropna().unique().tolist())
companias_unicas = sorted(df_fc['COMPANIA'].dropna().unique().tolist())
ciudades_unicas = sorted(df_fc['CIUDAD'].dropna().unique().tolist())

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_num = df_fc[df_fc['AÑO'] == ultimo_anio]['FECHA'].dt.month.max()
ultimo_mes_txt = meses_dict.get(ultimo_mes_num, 'Diciembre')

# 3. INTERFAZ (UI)
dd_anio_ciu = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_ciu = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_ciu = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_cias_ciu = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '120px'})
sel_ciudades = widgets.SelectMultiple(options=ciudades_unicas, value=[], description='🏙️ Ciudades:', layout={'width': '350px', 'height': '120px'})
sel_ramos_ciu = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})

btn_ciu = widgets.Button(description='Generar Tablero Ciudad', button_style='primary', icon='building', layout={'width': '300px', 'height': '40px'})

ui_ciu = widgets.VBox([
    widgets.HBox([dd_anio_ciu, dd_mes_ciu, sel_anios_atras_ciu]),
    widgets.HBox([sel_cias_ciu, sel_ciudades]),
    sel_ramos_ciu,
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Filtra por el acumulado YTD al mes seleccionado. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_ciu
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})
out_ciu = widgets.Output()

# 4. MOTOR ANALÍTICO
def generar_tablero_ciudad(b):
    with out_ciu:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_ciu.value, list(sel_anios_atras_ciu.value)
        companias_obj, ciu_obj = list(sel_cias_ciu.value), list(sel_ciudades.value)
        ramo_obj = list(sel_ramos_ciu.value)
        mes_str = dd_mes_ciu.value
        mes_num = meses_inversos[mes_str]

        if not companias_obj or not ciu_obj or not ramo_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        display(Markdown(f"# 📊 Análisis YTD por Ciudad (Acumulado a {mes_str})"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            df_fc_fil = df_fc[(df_fc['AÑO'].isin([anio_base, anio_ant])) &
                              (df_fc['FECHA'].dt.month == mes_num) &
                              (df_fc['CIUDAD'].isin(ciu_obj)) &
                              (df_fc['RAMOS'].isin(ramo_obj))].copy()
            if df_fc_fil.empty:
                display(Markdown(f"⚠️ No hay datos para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'CIUDAD'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = 0

            todos_indices = sorted(list(mkt.index))

            cols = []
            for cia in ['MERCADO'] + companias_obj:
                cols.extend([(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')])
                if cia != 'MERCADO': cols.append((cia, 'Veces'))

            cols_mi = pd.MultiIndex.from_tuples(cols)
            res = pd.DataFrame(index=todos_indices, columns=cols_mi)

            res[('MERCADO', lbl_anterior)] = mkt.get(anio_ant, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
            res[('MERCADO', lbl_actual)] = mkt.get(anio_base, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
            res[('MERCADO', 'Var')] = (res[('MERCADO', lbl_actual)] / res[('MERCADO', lbl_anterior)].replace(0, np.nan)) - 1

            for cia in companias_obj:
                df_cia = df_fc_fil[df_fc_fil['COMPANIA'] == cia]
                cia_agg = df_cia.groupby(['AÑO', 'CIUDAD'])['TOTAL'].sum().unstack('AÑO').fillna(0)

                res[(cia, lbl_anterior)] = cia_agg.get(anio_ant, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
                res[(cia, lbl_actual)] = cia_agg.get(anio_base, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
                res[(cia, 'Var')] = (res[(cia, lbl_actual)] / res[(cia, lbl_anterior)].replace(0, np.nan)) - 1
                res[(cia, 'Veces')] = res[(cia, 'Var')] / res[('MERCADO', 'Var')].replace(0, np.nan).abs()

            orden_filas = sorted([r for r in ciu_obj if r in res.index])
            res = res.loc[orden_filas]

            tot_series = res.sum()
            for cia in ['MERCADO'] + companias_obj:
                ant_tot = tot_series.get((cia, lbl_anterior), 0)
                act_tot = tot_series.get((cia, lbl_actual), 0)
                tot_series[(cia, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

                if cia != 'MERCADO':
                    v_m_tot = tot_series.get(('MERCADO', 'Var'), np.nan)
                    tot_series[(cia, 'Veces')] = (tot_series.get((cia, 'Var'), np.nan) / abs(v_m_tot)) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan

            res.loc['TOTAL'] = tot_series

            for cia in ['MERCADO'] + companias_obj:
                mask = (res[(cia, lbl_anterior)] == 0) & (res[(cia, lbl_actual)] == 0) & (res.index != 'TOTAL')
                if mask.any():
                    cols_to_cast = [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')]
                    if cia != 'MERCADO': cols_to_cast.append((cia, 'Veces'))
                    for col in cols_to_cast: res[col] = res[col].astype(object)
                    res.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                    res.loc[mask, (cia, 'Var')] = "-"
                    if cia != 'MERCADO': res.loc[mask, (cia, 'Veces')] = "-"

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
            def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
            def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            # ==================================================================
            # CORRECCIÓN DEFINITIVA DE ESTILOS (Soporte Pandas 2.0+)
            # ==================================================================
            def estilo_celdas(data):
                # Añadimos dtype=object para permitir textos y números mixtos sin fallar
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for cia in ['MERCADO'] + companias_obj:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        # Uso de listas nativas (list comprehension) en lugar de np.where
                        df_st[(cia, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (cia, 'Veces') in data.columns:
                        v_veces = pd.to_numeric(data[(cia, 'Veces')], errors='coerce')
                        df_st[(cia, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                 'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                 'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_veces]

                    m_sin = data[(cia, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (cia, 'Veces') in data.columns: df_st.loc[m_sin, (cia, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    # Forzamos .astype(str) para evitar choque de tipos con la fila TOTAL
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

            gc.collect()

btn_ciu.on_click(generar_tablero_ciudad)
display(ui_ciu, out_ciu)

# Por sucursal

In [ ]:
# ==============================================================================
# BLOQUE 8.5: TABLERO YTD - SUCURSAL (VERSIÓN DINÁMICA DE COMPAÑÍAS)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# 1. CARGA DE DATOS
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# 2. PARÁMETROS PARA WIDGETS
anios_disponibles = sorted(df_fc['AÑO'].dropna().unique().tolist())
ramos_unicos = sorted(df_fc['RAMOS'].dropna().unique().tolist())
companias_unicas = sorted(df_fc['COMPANIA'].dropna().unique().tolist())
sucursales_unicas = sorted(df_fc['SUCURSAL'].dropna().unique().tolist())

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_num = df_fc[df_fc['AÑO'] == ultimo_anio]['FECHA'].dt.month.max()
ultimo_mes_txt = meses_dict.get(ultimo_mes_num, 'Diciembre')

# 3. INTERFAZ (UI)
dd_anio_suc = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_suc = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_suc = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_cias_suc = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '120px'})
sel_sucursales = widgets.SelectMultiple(options=sucursales_unicas, value=[], description='📍 Sucursales:', layout={'width': '350px', 'height': '120px'})
sel_ramos_suc = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})

btn_suc = widgets.Button(description='Generar Tablero Sucursal', button_style='primary', icon='sitemap', layout={'width': '300px', 'height': '40px'})

ui_suc = widgets.VBox([
    widgets.HBox([dd_anio_suc, dd_mes_suc, sel_anios_atras_suc]),
    widgets.HBox([sel_cias_suc, sel_sucursales]),
    sel_ramos_suc,
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Filtra por el acumulado YTD al mes seleccionado. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_suc
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})
out_suc = widgets.Output()

# 4. MOTOR ANALÍTICO
def generar_tablero_sucursal(b):
    with out_suc:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_suc.value, list(sel_anios_atras_suc.value)
        companias_obj, suc_obj = list(sel_cias_suc.value), list(sel_sucursales.value)
        ramo_obj = list(sel_ramos_suc.value)
        mes_str = dd_mes_suc.value
        mes_num = meses_inversos[mes_str]

        if not companias_obj or not suc_obj or not ramo_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        display(Markdown(f"# 📊 Análisis YTD por Sucursal (Acumulado a {mes_str})"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            df_fc_fil = df_fc[(df_fc['AÑO'].isin([anio_base, anio_ant])) &
                              (df_fc['FECHA'].dt.month == mes_num) &
                              (df_fc['SUCURSAL'].isin(suc_obj)) &
                              (df_fc['RAMOS'].isin(ramo_obj))].copy()
            if df_fc_fil.empty:
                display(Markdown(f"⚠️ No hay datos para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'SUCURSAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = 0

            todos_indices = sorted(list(mkt.index))

            cols = []
            for cia in ['MERCADO'] + companias_obj:
                cols.extend([(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')])
                if cia != 'MERCADO': cols.append((cia, 'Veces'))

            cols_mi = pd.MultiIndex.from_tuples(cols)
            res = pd.DataFrame(index=todos_indices, columns=cols_mi)

            res[('MERCADO', lbl_anterior)] = mkt.get(anio_ant, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
            res[('MERCADO', lbl_actual)] = mkt.get(anio_base, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
            res[('MERCADO', 'Var')] = (res[('MERCADO', lbl_actual)] / res[('MERCADO', lbl_anterior)].replace(0, np.nan)) - 1

            for cia in companias_obj:
                df_cia = df_fc_fil[df_fc_fil['COMPANIA'] == cia]
                cia_agg = df_cia.groupby(['AÑO', 'SUCURSAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)

                res[(cia, lbl_anterior)] = cia_agg.get(anio_ant, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
                res[(cia, lbl_actual)] = cia_agg.get(anio_base, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
                res[(cia, 'Var')] = (res[(cia, lbl_actual)] / res[(cia, lbl_anterior)].replace(0, np.nan)) - 1
                # ⚠️ CORRECCIÓN ABS()
                res[(cia, 'Veces')] = res[(cia, 'Var')] / res[('MERCADO', 'Var')].replace(0, np.nan).abs()

            orden_filas = sorted([r for r in suc_obj if r in res.index])
            res = res.loc[orden_filas]

            tot_series = res.sum()
            for cia in ['MERCADO'] + companias_obj:
                ant_tot = tot_series.get((cia, lbl_anterior), 0)
                act_tot = tot_series.get((cia, lbl_actual), 0)
                tot_series[(cia, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

                if cia != 'MERCADO':
                    v_m_tot = tot_series.get(('MERCADO', 'Var'), np.nan)
                    # ⚠️ CORRECCIÓN ABS() EN TOTAL
                    tot_series[(cia, 'Veces')] = (tot_series.get((cia, 'Var'), np.nan) / abs(v_m_tot)) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan

            res.loc['TOTAL'] = tot_series

            for cia in ['MERCADO'] + companias_obj:
                mask = (res[(cia, lbl_anterior)] == 0) & (res[(cia, lbl_actual)] == 0) & (res.index != 'TOTAL')
                if mask.any():
                    cols_to_cast = [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')]
                    if cia != 'MERCADO': cols_to_cast.append((cia, 'Veces'))
                    for col in cols_to_cast: res[col] = res[col].astype(object)
                    res.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                    res.loc[mask, (cia, 'Var')] = "-"
                    if cia != 'MERCADO': res.loc[mask, (cia, 'Veces')] = "-"

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
            def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
            def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            # ==================================================================
            # CORRECCIÓN DEFINITIVA DE ESTILOS (Soporte Pandas 2.0+)
            # ==================================================================
            def estilo_celdas(data):
                # Añadimos dtype=object para permitir textos y números mixtos sin fallar
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for cia in ['MERCADO'] + companias_obj:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        # Uso de listas nativas (list comprehension) en lugar de np.where
                        df_st[(cia, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (cia, 'Veces') in data.columns:
                        v_veces = pd.to_numeric(data[(cia, 'Veces')], errors='coerce')
                        df_st[(cia, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                 'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                 'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_veces]

                    m_sin = data[(cia, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (cia, 'Veces') in data.columns: df_st.loc[m_sin, (cia, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    # Forzamos .astype(str) para evitar choque de tipos con la fila TOTAL
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

            gc.collect()

btn_suc.on_click(generar_tablero_sucursal)
display(ui_suc, out_suc)

# Por regional

In [ ]:
# ==============================================================================
# BLOQUE 8.3: TABLERO YTD - REGIONAL (VERSIÓN DINÁMICA DE COMPAÑÍAS)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# 1. CARGA DE DATOS
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# 2. PARÁMETROS PARA WIDGETS
anios_disponibles = sorted(df_fc['AÑO'].dropna().unique().tolist())
ramos_unicos = sorted(df_fc['RAMOS'].dropna().unique().tolist())
companias_unicas = sorted(df_fc['COMPANIA'].dropna().unique().tolist())
regionales_unicas = sorted(df_fc['REGIONAL'].dropna().unique().tolist())

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_num = df_fc[df_fc['AÑO'] == ultimo_anio]['FECHA'].dt.month.max()
ultimo_mes_txt = meses_dict.get(ultimo_mes_num, 'Diciembre')

# 3. INTERFAZ (UI)
dd_anio_reg = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_reg = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_reg = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_cias_reg = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '120px'})
sel_reg_reg = widgets.SelectMultiple(options=regionales_unicas, value=[], description='🗺️ Regionales:', layout={'width': '350px', 'height': '120px'})
sel_ramos_reg = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})

btn_reg = widgets.Button(description='Generar Tablero Regional', button_style='primary', icon='map', layout={'width': '300px', 'height': '40px'})

ui_reg = widgets.VBox([
    widgets.HBox([dd_anio_reg, dd_mes_reg, sel_anios_atras_reg]),
    widgets.HBox([sel_cias_reg, sel_reg_reg]),
    sel_ramos_reg,
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Filtra por el acumulado YTD al mes seleccionado. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_reg
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})
out_reg = widgets.Output()

# 4. MOTOR ANALÍTICO
def generar_tablero_regional(b):
    with out_reg:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_reg.value, list(sel_anios_atras_reg.value)
        companias_obj, reg_obj = list(sel_cias_reg.value), list(sel_reg_reg.value)
        ramo_obj = list(sel_ramos_reg.value)
        mes_str = dd_mes_reg.value
        mes_num = meses_inversos[mes_str]

        if not companias_obj or not reg_obj or not ramo_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        display(Markdown(f"# 📊 Análisis YTD por Regional (Acumulado a {mes_str})"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            df_fc_fil = df_fc[(df_fc['AÑO'].isin([anio_base, anio_ant])) &
                              (df_fc['FECHA'].dt.month == mes_num) &
                              (df_fc['REGIONAL'].isin(reg_obj)) &
                              (df_fc['RAMOS'].isin(ramo_obj))].copy()
            if df_fc_fil.empty:
                display(Markdown(f"⚠️ No hay datos para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'REGIONAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = 0

            todos_indices = sorted(list(mkt.index))

            cols = []
            for cia in ['MERCADO'] + companias_obj:
                cols.extend([(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')])
                if cia != 'MERCADO': cols.append((cia, 'Veces'))

            cols_mi = pd.MultiIndex.from_tuples(cols)
            res = pd.DataFrame(index=todos_indices, columns=cols_mi)

            # MERCADO
            res[('MERCADO', lbl_anterior)] = mkt.get(anio_ant, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
            res[('MERCADO', lbl_actual)] = mkt.get(anio_base, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
            res[('MERCADO', 'Var')] = (res[('MERCADO', lbl_actual)] / res[('MERCADO', lbl_anterior)].replace(0, np.nan)) - 1

            # COMPAÑÍAS
            for cia in companias_obj:
                df_cia = df_fc_fil[df_fc_fil['COMPANIA'] == cia]
                cia_agg = df_cia.groupby(['AÑO', 'REGIONAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)

                res[(cia, lbl_anterior)] = cia_agg.get(anio_ant, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
                res[(cia, lbl_actual)] = cia_agg.get(anio_base, pd.Series(0, index=todos_indices)).reindex(todos_indices).fillna(0)
                res[(cia, 'Var')] = (res[(cia, lbl_actual)] / res[(cia, lbl_anterior)].replace(0, np.nan)) - 1
                # ⚠️ CORRECCIÓN ABS()
                res[(cia, 'Veces')] = res[(cia, 'Var')] / res[('MERCADO', 'Var')].replace(0, np.nan).abs()

            orden_filas = sorted([r for r in reg_obj if r in res.index])
            res = res.loc[orden_filas]

            # TOTAL
            tot_series = res.sum()
            for cia in ['MERCADO'] + companias_obj:
                ant_tot = tot_series.get((cia, lbl_anterior), 0)
                act_tot = tot_series.get((cia, lbl_actual), 0)
                tot_series[(cia, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

                if cia != 'MERCADO':
                    v_m_tot = tot_series.get(('MERCADO', 'Var'), np.nan)
                    # ⚠️ CORRECCIÓN ABS() EN TOTAL
                    tot_series[(cia, 'Veces')] = (tot_series.get((cia, 'Var'), np.nan) / abs(v_m_tot)) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan

            res.loc['TOTAL'] = tot_series

            for cia in ['MERCADO'] + companias_obj:
                mask = (res[(cia, lbl_anterior)] == 0) & (res[(cia, lbl_actual)] == 0) & (res.index != 'TOTAL')
                if mask.any():
                    cols_to_cast = [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')]
                    if cia != 'MERCADO': cols_to_cast.append((cia, 'Veces'))
                    for col in cols_to_cast: res[col] = res[col].astype(object)
                    res.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                    res.loc[mask, (cia, 'Var')] = "-"
                    if cia != 'MERCADO': res.loc[mask, (cia, 'Veces')] = "-"

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
            def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
            def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            # ==================================================================
            # CORRECCIÓN DEFINITIVA DE ESTILOS (Soporte Pandas 2.0+)
            # ==================================================================
            def estilo_celdas(data):
                # Añadimos dtype=object para permitir textos y números mixtos sin fallar
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for cia in ['MERCADO'] + companias_obj:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        # Uso de listas nativas (list comprehension) en lugar de np.where
                        df_st[(cia, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (cia, 'Veces') in data.columns:
                        v_veces = pd.to_numeric(data[(cia, 'Veces')], errors='coerce')
                        df_st[(cia, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                 'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                 'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_veces]

                    m_sin = data[(cia, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (cia, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (cia, 'Veces') in data.columns: df_st.loc[m_sin, (cia, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    # Forzamos .astype(str) para evitar choque de tipos con la fila TOTAL
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

            gc.collect()

btn_reg.on_click(generar_tablero_regional)
display(ui_reg, out_reg)

#Crecimientos

In [ ]:
# ==============================================================================
# BLOQUE 17: COMPARATIVO DIRECTO DE CRECIMIENTO VS MERCADO (INDEPENDIENTE)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS (Base Acumulada YTD)
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted(df_fc['AÑO'].dropna().unique().tolist())
companias_unicas = sorted(df_fc['COMPANIA'].dropna().unique().tolist())
ramos_unicos = sorted(df_fc['RAMOS'].dropna().unique().tolist())

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_num = df_fc[df_fc['AÑO'] == ultimo_anio]['FECHA'].dt.month.max()
ultimo_mes_txt = meses_dict.get(ultimo_mes_num, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_comp = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_comp = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_comp = widgets.Dropdown(options=[1, 2, 3, 4, 5, 10], value=1, description='⏳ Años Atrás:', layout={'width': '200px'})

sel_cias_comp = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '150px'})
sel_ramos_comp = widgets.SelectMultiple(options=ramos_unicos, value=ramos_unicos, description='📋 Ramos:', layout={'width': '350px', 'height': '150px'})

btn_comp = widgets.Button(description='Generar Comparativo', button_style='success', icon='bar-chart', layout={'width': '300px', 'height': '40px'})
out_comp = widgets.Output()

ui_comp = widgets.VBox([
    widgets.HBox([dd_anio_comp, dd_mes_comp, sel_anios_atras_comp]),
    widgets.HBox([sel_cias_comp, sel_ramos_comp]),
    widgets.HTML("<em style='color:gray; font-size:11px;'>* Selecciona las compañías que deseas comparar contra la fila fija del Mercado.</em>"),
    btn_comp
], layout={'border': '1px solid #ddd', 'padding': '15px', 'background-color': '#fdfdfd', 'border-radius': '5px'})

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_comparativo_directo(b):
    with out_comp:
        clear_output(wait=True)

        anio_base = dd_anio_comp.value
        anio_ant = anio_base - sel_anios_atras_comp.value
        mes_num = meses_inversos[dd_mes_comp.value]
        cias_sel = list(sel_cias_comp.value)
        ramos_sel = list(sel_ramos_comp.value)

        if not cias_sel or not ramos_sel:
            display(Markdown("⚠️ **Aviso:** Debes seleccionar al menos una compañía y un ramo para comparar."))
            return

        # FILTRO YTD DIRECTO A LA BASE ACUMULADA
        df_fil = df_fc[(df_fc['AÑO'].isin([anio_base, anio_ant])) &
                       (df_fc['FECHA'].dt.month == mes_num) &
                       (df_fc['RAMOS'].isin(ramos_sel))].copy()

        if df_fil.empty:
            display(Markdown(f"⚠️ **No hay datos para {anio_ant} vs {anio_base} en el mes de {dd_mes_comp.value}.**"))
            return

        # 1. CÁLCULO DEL MERCADO (La base fija)
        mkt_ant = df_fil[df_fil['AÑO'] == anio_ant]['TOTAL'].sum()
        mkt_act = df_fil[df_fil['AÑO'] == anio_base]['TOTAL'].sum()
        mkt_var = (mkt_act / mkt_ant - 1) if mkt_ant else np.nan

        datos = []
        # Agregamos la fila del mercado primero
        datos.append({
            'COMPAÑÍA': '*** TOTAL MERCADO ***',
            f'Primas Actuales ({anio_base})': mkt_act,
            'Crecimiento (%)': mkt_var
        })

        # 2. CÁLCULO DE LAS COMPAÑÍAS SELECCIONADAS
        for cia in cias_sel:
            df_cia = df_fil[df_fil['COMPANIA'] == cia]
            c_ant = df_cia[df_cia['AÑO'] == anio_ant]['TOTAL'].sum()
            c_act = df_cia[df_cia['AÑO'] == anio_base]['TOTAL'].sum()

            if c_ant == 0 and c_act == 0:
                c_var = np.nan
                c_act = "Sin registros"
            else:
                c_var = (c_act / c_ant - 1) if c_ant else np.nan

            datos.append({
                'COMPAÑÍA': cia,
                f'Primas Actuales ({anio_base})': c_act,
                'Crecimiento (%)': c_var
            })

        df_res = pd.DataFrame(datos).set_index('COMPAÑÍA')

        display(Markdown(f"### 📊 Crecimiento Directo: {anio_base} vs {anio_ant} (Acumulado a {dd_mes_comp.value})"))

        # FUNCIONES DE FORMATO Y ESTILO
        def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
        def fmt_p(v): return "-" if pd.isna(v) else f"{v:+.2%}"

        def aplicar_estilos_rk(data):
            df_st = pd.DataFrame('', index=data.index, columns=data.columns)

            # Estilo Mercado (Fila Fija)
            if '*** TOTAL MERCADO ***' in df_st.index:
                df_st.loc['*** TOTAL MERCADO ***'] = 'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B; border-bottom: 2px solid #0F753B; color: black;'

            # Estilo Crecimiento (Semáforo Verde/Rojo)
            if 'Crecimiento (%)' in data.columns:
                v_var = pd.to_numeric(data['Crecimiento (%)'], errors='coerce')
                df_st['Crecimiento (%)'] = np.where(v_var > 0, ' color: #1E8449; font-weight: bold;',
                                            np.where(v_var < 0, ' color: #E74C3C; font-weight: bold;', ' color: gray;'))

            # Estilo "Sin registros"
            for idx, row in data.iterrows():
                if row.iloc[0] == "Sin registros":
                    df_st.loc[idx] = 'color: #bdc3c7; font-style: italic; text-align: center;'

            return df_st

        # RENDERIZAR TABLA
        st_res = df_res.style.format({f'Primas Actuales ({anio_base})': fmt_m, 'Crecimiento (%)': fmt_p}, na_rep="-")\
                .apply(aplicar_estilos_rk, axis=None)\
                .set_table_styles([
                    {'selector': 'th', 'props': [('background-color', '#0F753B'), ('color', 'white'), ('font-size', '12px'), ('text-align', 'center'), ('padding', '8px')]},
                    {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '12px'), ('padding', '8px')]},
                    {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('background-color', '#f4f4f4'), ('color', 'black')]}
                ])
        display(st_res)

        gc.collect()

btn_comp.on_click(generar_comparativo_directo)
display(ui_comp, out_comp)



#Bloque maestro

In [ ]:
# ==============================================================================
# BLOQUE 16: DASHBOARD MAESTRO TIPO POWER BI (RESUMEN EJECUTIVO ACTUALIZADO)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output, HTML
from dateutil.relativedelta import relativedelta
import warnings
import gc
warnings.filterwarnings('ignore')

# 1. Extracción de Parámetros
# ------------------------------------------------------------------------------
companias_unicas = sorted(df_maestro['COMPANIA'].dropna().unique().tolist())
ramos_unicos = sorted(df_maestro['RAMOS'].dropna().unique().tolist())
regionales_unicas = sorted(df_maestro['REGIONAL'].dropna().unique().tolist())
fechas_disponibles = sorted(df_maestro['FECHA'].dropna().unique())
anios_disponibles = sorted(df_maestro['AÑO'].dropna().unique().tolist())

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}
meses_nombres = list(meses_dict.values())

COLOR_VERDE = '#0F753B'
COLOR_CREMA = '#FCE49C'

# 2. Construcción de UI
# ------------------------------------------------------------------------------
ultimo_anio = fechas_disponibles[-1].year
ultimo_mes_txt = meses_dict[fechas_disponibles[-1].month]

dropdown_anio_inicio = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='Inicio - Año:', layout={'width': '160px'})
dropdown_mes_inicio = widgets.Dropdown(options=meses_nombres, value=meses_dict[1], description='Mes:', layout={'width': '140px'})
dropdown_anio_fin = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='Final - Año:', layout={'width': '160px'})
dropdown_mes_fin = widgets.Dropdown(options=meses_nombres, value=ultimo_mes_txt, description='Mes:', layout={'width': '140px'})
input_meses = widgets.BoundedIntText(value=1, min=1, max=1000, description='Cant. Meses:', layout={'width': '150px'})
opciones_anios_atras = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
select_anios_atras = widgets.SelectMultiple(options=opciones_anios_atras, value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
mensaje_alerta = widgets.HTML(value="")

def actualizar_meses_desde_fechas(*args):
    input_meses.unobserve(actualizar_fechas_desde_meses, 'value')
    fecha_ini = pd.Timestamp(year=dropdown_anio_inicio.value, month=meses_inversos[dropdown_mes_inicio.value], day=1)
    fecha_fin = pd.Timestamp(year=dropdown_anio_fin.value, month=meses_inversos[dropdown_mes_fin.value], day=1)
    if fecha_fin >= fecha_ini:
        diff_meses = (fecha_fin.year - fecha_ini.year) * 12 + (fecha_fin.month - fecha_ini.month) + 1
        input_meses.value = diff_meses
        anios_salto_max = max(select_anios_atras.value) if select_anios_atras.value else 0
        fecha_ini_ant_max = fecha_ini - relativedelta(years=anios_salto_max)
        if anios_salto_max > 0 and fecha_ini_ant_max < fechas_disponibles[0]:
            mensaje_alerta.value = f"<span style='color:#E74C3C; font-weight:bold;'>⚠️ Alerta: No hay historia para comparar {anios_salto_max} año(s) atrás.</span>"
        else:
            mensaje_alerta.value = f"<span style='color:#1E8449; font-weight:bold;'>✅ Periodo válido. (Base YTD usará corte a {dropdown_mes_fin.value})</span>"
    else: mensaje_alerta.value = ""
    input_meses.observe(actualizar_fechas_desde_meses, 'value')

def actualizar_fechas_desde_meses(*args):
    dropdown_anio_fin.unobserve(actualizar_meses_desde_fechas, 'value')
    dropdown_mes_fin.unobserve(actualizar_meses_desde_fechas, 'value')
    fecha_ini = pd.Timestamp(year=dropdown_anio_inicio.value, month=meses_inversos[dropdown_mes_inicio.value], day=1)
    nueva_fecha_fin = fecha_ini + relativedelta(months=input_meses.value - 1)
    if nueva_fecha_fin > fechas_disponibles[-1]: nueva_fecha_fin = fechas_disponibles[-1]
    dropdown_anio_fin.value = nueva_fecha_fin.year
    dropdown_mes_fin.value = meses_dict[nueva_fecha_fin.month]
    dropdown_anio_fin.observe(actualizar_meses_desde_fechas, 'value')
    dropdown_mes_fin.observe(actualizar_meses_desde_fechas, 'value')
    actualizar_meses_desde_fechas()

dropdown_anio_inicio.observe(actualizar_meses_desde_fechas, 'value')
dropdown_mes_inicio.observe(actualizar_meses_desde_fechas, 'value')
dropdown_anio_fin.observe(actualizar_meses_desde_fechas, 'value')
dropdown_mes_fin.observe(actualizar_meses_desde_fechas, 'value')
input_meses.observe(actualizar_fechas_desde_meses, 'value')
select_anios_atras.observe(actualizar_meses_desde_fechas, 'value')
actualizar_meses_desde_fechas()

select_ramos = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '250px', 'height': '120px'})
select_companias = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Cías:', layout={'width': '250px', 'height': '120px'})
select_regionales = widgets.SelectMultiple(options=regionales_unicas, value=[], description='🗺️ Reg:', layout={'width': '250px', 'height': '120px'})

boton_generar = widgets.Button(description='Generar Dashboard', button_style='success', icon='tachometer', layout={'width': '300px', 'height': '40px'})
out_main = widgets.Output()

# 3. Funciones de Apoyo Matemático y Formato
# ------------------------------------------------------------------------------
def get_pivot(df_base, dimension, entidades, lbl_actual, lbl_anterior):
    df_m = df_base.groupby(['ETIQUETA_TIEMPO', dimension])['TOTAL'].sum().reset_index()
    df_m['COMPANIA'] = 'MERCADO'
    df_c = df_base[df_base['COMPANIA'].isin(entidades)].groupby(['ETIQUETA_TIEMPO', 'COMPANIA', dimension])['TOTAL'].sum().reset_index()

    df_con = pd.concat([df_c, df_m], ignore_index=True)
    df_tot = df_con.groupby(['ETIQUETA_TIEMPO', 'COMPANIA'])['TOTAL'].sum().reset_index()
    df_tot[dimension] = 'TOTAL'
    df_fin = pd.concat([df_con, df_tot], ignore_index=True)

    df_piv = df_fin.pivot_table(index=dimension, columns=['COMPANIA', 'ETIQUETA_TIEMPO'], values='TOTAL', aggfunc='sum').fillna(0)

    frames = []
    ord_c = ['MERCADO'] + [c for c in entidades]

    a_mkt = df_piv.get(('MERCADO', lbl_anterior), pd.Series(0, index=df_piv.index))
    h_mkt = df_piv.get(('MERCADO', lbl_actual), pd.Series(0, index=df_piv.index))
    v_mkt = pd.Series(np.nan, index=df_piv.index)
    v_mkt[a_mkt != 0] = (h_mkt[a_mkt != 0] / a_mkt[a_mkt != 0]) - 1

    for cia in ord_c:
        if cia not in df_piv.columns.get_level_values(0): continue
        ayer = df_piv.get((cia, lbl_anterior), pd.Series(0, index=df_piv.index))
        hoy = df_piv.get((cia, lbl_actual), pd.Series(0, index=df_piv.index))
        var = pd.Series(np.nan, index=df_piv.index)
        var[ayer != 0] = (hoy[ayer != 0] / ayer[ayer != 0]) - 1

        if cia == 'MERCADO':
            frames.append(pd.DataFrame({(cia, lbl_anterior): ayer, (cia, lbl_actual): hoy, (cia, 'Var'): var}))
        else:
            veces = pd.Series(np.nan, index=df_piv.index)
            m_v = v_mkt.notna() & (v_mkt != 0)
            veces[m_v] = var[m_v] / v_mkt[m_v].abs()
            frames.append(pd.DataFrame({(cia, lbl_anterior): ayer, (cia, lbl_actual): hoy, (cia, 'Var'): var, (cia, 'Veces'): veces}))

    if not frames: return pd.DataFrame(), []
    tab = pd.concat(frames, axis=1)

    for cia in ord_c:
        if (cia, lbl_anterior) in tab.columns:
            mask = (tab[(cia, lbl_anterior)] == 0) & (tab[(cia, lbl_actual)] == 0) & (tab.index != 'TOTAL')
            if mask.any():
                cols_to_cast = [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var')]
                if (cia, 'Veces') in tab.columns: cols_to_cast.append((cia, 'Veces'))
                for col in cols_to_cast:
                    tab[col] = tab[col].astype(object)
                tab.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                tab.loc[mask, (cia, 'Var')] = "-"
                if (cia, 'Veces') in tab.columns: tab.loc[mask, (cia, 'Veces')] = "-"

    return tab, ord_c

def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

def aplicar_estilos(data, ord_c, lbl_a, lbl_h):
    # APLICADO dtype=object para asegurar compatibilidad con strings en Pandas 2.0+
    df_e = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
    for c in ord_c:
        if (c, 'Var') in data.columns:
            vc = pd.to_numeric(data[(c, 'Var')], errors='coerce')
            df_e[(c, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in vc]

        if (c, 'Veces') in data.columns:
            v_vec = pd.to_numeric(data[(c, 'Veces')], errors='coerce')
            df_e[(c, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                 'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                 'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_vec]

        m_sin = data[(c, lbl_h)] == "Sin registros"
        if m_sin.any():
            df_e.loc[m_sin, (c, lbl_a)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
            df_e.loc[m_sin, (c, lbl_h)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
            df_e.loc[m_sin, (c, 'Var')] = 'text-align: center; color: #bdc3c7;'
            if (c, 'Veces') in data.columns: df_e.loc[m_sin, (c, 'Veces')] = 'text-align: center; color: #bdc3c7;'

    if 'TOTAL' in df_e.index:
        df_e.loc['TOTAL'] = df_e.loc['TOTAL'].astype(str) + f' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid {COLOR_VERDE};'
    return df_e

tbl_styles = [
    {'selector': 'th.col_heading.level0', 'props': [(f'background-color', COLOR_VERDE), ('color', 'white'), ('text-align', 'center'), ('font-size', '11px'), ('border-right', '1px solid white')]},
    {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '10px')]},
    {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('font-size', '10px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '11px')]},
    {'selector': 'td:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]},
    {'selector': 'th.col_heading.level1:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]}
]

def header_html(titulo):
    return f"<div style='background-color:{COLOR_CREMA}; padding:4px; text-align:center; border:1px solid {COLOR_VERDE}; margin-bottom: 5px;'><h4 style='margin:0; color:#333; font-size:12px; font-weight:bold;'>{titulo}</h4></div>"

# 4. Motor Principal (Dashboard Layout)
# ------------------------------------------------------------------------------
def generar_tablero_maestro(b):
    with out_main:
        clear_output(wait=True)

        anio_fin, mes_fin = dropdown_anio_fin.value, dropdown_mes_fin.value
        anios_atras_lista = list(select_anios_atras.value)
        ramos_obj = list(select_ramos.value)
        cias_obj = list(select_companias.value)
        reg_obj = list(select_regionales.value)

        if not ramos_obj or not cias_obj or not reg_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        fecha_fin = pd.Timestamp(year=anio_fin, month=meses_inversos[mes_fin], day=1)

        for anios_salto in sorted(anios_atras_lista):
            fecha_fin_ant = fecha_fin - relativedelta(years=anios_salto)
            if fecha_fin_ant < fechas_disponibles[0]: continue

            lbl_actual, lbl_anterior = f"Cierre ({fecha_fin.year})", f"Anterior ({fecha_fin_ant.year})"

            df_base = df_maestro[(df_maestro['FECHA'].isin([fecha_fin, fecha_fin_ant])) &
                                 (df_maestro['RAMOS'].isin(ramos_obj))].copy()

            df_base_reg = df_base[df_base['REGIONAL'].isin(reg_obj)].copy()
            if df_base_reg.empty: continue

            df_base['ETIQUETA_TIEMPO'] = np.where(df_base['FECHA'] == fecha_fin, lbl_actual, lbl_anterior)
            df_base_reg['ETIQUETA_TIEMPO'] = np.where(df_base_reg['FECHA'] == fecha_fin, lbl_actual, lbl_anterior)

            # CÁLCULOS ESTÁNDAR
            tab_reg, cols_r = get_pivot(df_base_reg, 'REGIONAL', cias_obj, lbl_actual, lbl_anterior)
            tab_ramo, cols_rm = get_pivot(df_base_reg, 'RAMOS', cias_obj, lbl_actual, lbl_anterior)

            # CONFIGURACIÓN DEL GRID (Ajustado a 6 salidas para 2 tablas abajo)
            box_layout = widgets.Layout(border=f'1px solid {COLOR_VERDE}', padding='5px', width='50%', height='350px', overflow='auto')

            out_hier = widgets.Output(layout=box_layout) # Top Left (Jerarquía)
            out_ml = widgets.Output(layout=box_layout)   # Top Right (Tabla Reg)
            out_mr = widgets.Output(layout=box_layout)   # Mid Left (Gráfica Veces)
            out_bl = widgets.Output(layout=box_layout)   # Mid Right (Tabla Ramo)
            out_br1 = widgets.Output(layout=box_layout)  # Bot Left (Top 10 Mayor)
            out_br2 = widgets.Output(layout=box_layout)  # Bot Right (Top 10 Menor)

            # --- TOP LEFT: Tabla Jerárquica ---
            with out_hier:
                display(HTML(header_html("DETALLE AÑO, REGIONAL Y SUCURSAL")))
                mkt_tot = df_base_reg.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                mkt_reg = df_base_reg.groupby(['REGIONAL', 'ETIQUETA_TIEMPO'])['TOTAL'].sum().unstack(fill_value=0)
                mkt_suc = df_base_reg.groupby(['REGIONAL', 'SUCURSAL', 'ETIQUETA_TIEMPO'])['TOTAL'].sum().unstack(fill_value=0)

                rows_hier = []
                row_anio = {'Nivel': 0, 'Agrupación': f'Año {fecha_fin.year}'}
                m_ant_tot = mkt_tot.get(lbl_anterior, 0)
                m_act_tot = mkt_tot.get(lbl_actual, 0)
                var_m_tot = (m_act_tot / m_ant_tot - 1) if m_ant_tot != 0 else np.nan
                row_anio['Var Mercado'] = var_m_tot

                for cia in cias_obj:
                    df_cia_t = df_base_reg[df_base_reg['COMPANIA'] == cia]
                    cia_t = df_cia_t.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                    c_ant_tot = cia_t.get(lbl_anterior, 0)
                    c_act_tot = cia_t.get(lbl_actual, 0)
                    var_c_tot = (c_act_tot / c_ant_tot - 1) if c_ant_tot != 0 else np.nan
                    row_anio[f'Var {cia}'] = var_c_tot
                    row_anio[f'Veces {cia}'] = (var_c_tot / abs(var_m_tot)) if (pd.notna(var_m_tot) and var_m_tot != 0) else np.nan
                rows_hier.append(row_anio)

                for reg in sorted(reg_obj):
                    if reg not in mkt_reg.index: continue
                    row_r = {'Nivel': 1, 'Agrupación': str(reg)}
                    m_ant_r = mkt_reg.loc[reg, lbl_anterior] if lbl_anterior in mkt_reg.columns else 0
                    m_act_r = mkt_reg.loc[reg, lbl_actual] if lbl_actual in mkt_reg.columns else 0
                    var_m_r = (m_act_r / m_ant_r - 1) if m_ant_r != 0 else np.nan
                    row_r['Var Mercado'] = var_m_r

                    for cia in cias_obj:
                        df_cia_r = df_base_reg[(df_base_reg['COMPANIA'] == cia) & (df_base_reg['REGIONAL'] == reg)]
                        cia_t_r = df_cia_r.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                        c_ant_r = cia_t_r.get(lbl_anterior, 0)
                        c_act_r = cia_t_r.get(lbl_actual, 0)
                        var_c_r = (c_act_r / c_ant_r - 1) if c_ant_r != 0 else np.nan
                        row_r[f'Var {cia}'] = var_c_r
                        row_r[f'Veces {cia}'] = (var_c_r / abs(var_m_r)) if (pd.notna(var_m_r) and var_m_r != 0) else np.nan
                    rows_hier.append(row_r)

                    sucursales = df_base_reg[df_base_reg['REGIONAL'] == reg]['SUCURSAL'].dropna().unique()
                    for suc in sorted(sucursales):
                        if (reg, suc) not in mkt_suc.index: continue
                        row_s = {'Nivel': 2, 'Agrupación': f"  ↳ {suc}"}
                        m_ant_s = mkt_suc.loc[(reg, suc), lbl_anterior] if lbl_anterior in mkt_suc.columns else 0
                        m_act_s = mkt_suc.loc[(reg, suc), lbl_actual] if lbl_actual in mkt_suc.columns else 0
                        var_m_s = (m_act_s / m_ant_s - 1) if m_ant_s != 0 else np.nan
                        row_s['Var Mercado'] = var_m_s

                        for cia in cias_obj:
                            df_cia_s = df_base_reg[(df_base_reg['COMPANIA'] == cia) & (df_base_reg['REGIONAL'] == reg) & (df_base_reg['SUCURSAL'] == suc)]
                            cia_t_s = df_cia_s.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                            c_ant_s = cia_t_s.get(lbl_anterior, 0)
                            c_act_s = cia_t_s.get(lbl_actual, 0)
                            var_c_s = (c_act_s / c_ant_s - 1) if c_ant_s != 0 else np.nan
                            row_s[f'Var {cia}'] = var_c_s
                            row_s[f'Veces {cia}'] = (var_c_s / abs(var_m_s)) if (pd.notna(var_m_s) and var_m_s != 0) else np.nan
                        rows_hier.append(row_s)

                df_hier = pd.DataFrame(rows_hier).set_index('Agrupación')
                f_d_hier = {'Var Mercado': fmt_p}
                for cia in cias_obj:
                    f_d_hier[f'Var {cia}'] = fmt_p
                    f_d_hier[f'Veces {cia}'] = fmt_v

                def estilo_jerarquia(data):
                    df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                    for idx, row in data.iterrows():
                        if row['Nivel'] == 0:
                            df_st.loc[idx] = f'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid {COLOR_VERDE}; border-bottom: 2px solid {COLOR_VERDE};'
                        elif row['Nivel'] == 1:
                            df_st.loc[idx] = 'font-weight: bold; background-color: #fcfcfc;'
                        elif row['Nivel'] == 2:
                            df_st.loc[idx] = 'font-style: italic; color: #555;'

                        for col in data.columns:
                            if 'Var' in col and col != 'Nivel':
                                v = pd.to_numeric(row[col], errors='coerce')
                                if pd.notna(v) and v > 0: df_st.loc[idx, col] = str(df_st.loc[idx, col]) + ' color: #1E8449;'
                                elif pd.notna(v) and v < 0: df_st.loc[idx, col] = str(df_st.loc[idx, col]) + ' color: #E74C3C;'

                            if 'Veces' in col:
                                v_veces = pd.to_numeric(row[col], errors='coerce')
                                estilo_actual = str(df_st.loc[idx, col])
                                if pd.isna(v_veces):
                                    df_st.loc[idx, col] = estilo_actual + ' color: #7F8C8D;'
                                elif v_veces > 1.8:
                                    df_st.loc[idx, col] = estilo_actual + ' color: #1E8449; font-weight: bold;'
                                elif v_veces >= 1.0:
                                    df_st.loc[idx, col] = estilo_actual + ' color: #F39C12; font-weight: bold;'
                                else:
                                    df_st.loc[idx, col] = estilo_actual + ' color: #E74C3C; font-weight: bold;'
                    return df_st

                st_hier = df_hier.style.format(f_d_hier, na_rep="-").apply(estilo_jerarquia, axis=None).hide(subset=['Nivel'], axis=1).set_table_styles([
                    {'selector': 'th.col_heading', 'props': [('background-color', COLOR_VERDE), ('color', 'white'), ('font-size', '11px')]},
                    {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '11px')]},
                    {'selector': 'th.row_heading', 'props': [('text-align', 'left')]}
                ])
                display(st_hier)

            # --- TOP RIGHT: Base Detalle Regional ---
            with out_ml:
                display(HTML(header_html("BASE DETALLE REGIONAL")))
                if not tab_reg.empty:
                    ord_r = ['TOTAL'] + sorted([r for r in reg_obj if r in tab_reg.index])
                    t_r = tab_reg.loc[ord_r]
                    fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in t_r.columns}
                    display(t_r.style.format(fc, na_rep="-").apply(aplicar_estilos, ord_c=cols_r, lbl_a=lbl_anterior, lbl_h=lbl_actual, axis=None).set_table_styles(tbl_styles))

            # --- MID LEFT: Veces por Ramo ---
            with out_mr:
                display(HTML(header_html("VECES POR RAMO")))
                if not tab_ramo.empty and cias_obj:
                    cia_ref = cias_obj[0]
                    if (cia_ref, 'Veces') in tab_ramo.columns:
                        # Filtrar los "-" explícitamente para evitar el TypeError
                        df_graf = tab_ramo.loc[[r for r in ramos_obj if r in tab_ramo.index]].dropna(subset=[(cia_ref, 'Veces')])
                        df_graf = df_graf[df_graf[(cia_ref, 'Veces')] != "-"]
                        if not df_graf.empty:
                            fig, ax = plt.subplots(figsize=(6, 2.5))
                            y_pos = np.arange(len(df_graf))
                            vals = pd.to_numeric(df_graf[(cia_ref, 'Veces')], errors='coerce').fillna(0).values
                            colors = ['#1E8449' if v > 1.8 else ('#F39C12' if v >= 1.0 else '#E74C3C') for v in vals]
                            ax.barh(y_pos, vals, align='center', color=colors, height=0.5)
                            ax.set_yticks(y_pos)
                            ax.set_yticklabels([str(x)[:12] for x in df_graf.index], fontsize=8)
                            ax.invert_yaxis()
                            ax.set_xlabel('Veces vs Mercado', fontsize=8)
                            for i, v in enumerate(vals):
                                ax.text(v + 0.1, i, f'{v:.2f}x', va='center', fontsize=8, color='gray')
                            ax.spines['top'].set_visible(False)
                            ax.spines['right'].set_visible(False)
                            plt.tight_layout()
                            plt.show()
                            plt.close(fig) # <--- OBLIGATORIO en Binder

            # --- MID RIGHT: % Var por Año y Ramo ---
            with out_bl:
                display(HTML(header_html("% VAR POR AÑO Y RAMO")))
                if not tab_ramo.empty:
                    ord_rm = ['TOTAL'] + sorted([r for r in ramos_obj if r in tab_ramo.index])
                    t_rm = tab_ramo.loc[ord_rm]
                    fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in t_rm.columns}
                    display(t_rm.style.format(fc, na_rep="-").apply(aplicar_estilos, ord_c=cols_rm, lbl_a=lbl_anterior, lbl_h=lbl_actual, axis=None).set_table_styles(tbl_styles))

            # --- BOTTOM: Rankings Top 10 Compañías ---
            df_rk_base = df_base.groupby(['COMPANIA', 'ETIQUETA_TIEMPO'])['TOTAL'].sum().unstack(fill_value=0)

            if lbl_actual in df_rk_base.columns and lbl_anterior in df_rk_base.columns:
                df_rk_base['Crecimiento (%)'] = (df_rk_base[lbl_actual] / df_rk_base[lbl_anterior].replace(0, np.nan)) - 1

                df_rk_valid = df_rk_base.replace([np.inf, -np.inf], np.nan).dropna(subset=['Crecimiento (%)'])

                df_top10 = df_rk_valid.sort_values('Crecimiento (%)', ascending=False).head(10)
                df_top10 = df_top10[[lbl_actual, 'Crecimiento (%)']].rename(columns={lbl_actual: 'Primas Actuales'})

                df_bot10 = df_rk_valid.sort_values('Crecimiento (%)', ascending=True).head(10)
                df_bot10 = df_bot10[[lbl_actual, 'Crecimiento (%)']].rename(columns={lbl_actual: 'Primas Actuales'})

                mkt_ant_rk = df_rk_base[lbl_anterior].sum()
                mkt_act_rk = df_rk_base[lbl_actual].sum()
                mkt_crec = (mkt_act_rk / mkt_ant_rk - 1) if mkt_ant_rk != 0 else np.nan

                df_top10.loc['*** TOTAL MERCADO ***'] = [mkt_act_rk, mkt_crec]
                df_bot10.loc['*** TOTAL MERCADO ***'] = [mkt_act_rk, mkt_crec]

                def c_rk(s): return ['color: #1E8449; font-weight: bold;' if v > 0 else 'color: #E74C3C; font-weight: bold;' if v < 0 else 'color: gray;' for v in s]

                def b_total(data):
                    df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                    if '*** TOTAL MERCADO ***' in df_st.index:
                        df_st.loc['*** TOTAL MERCADO ***'] = f'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid {COLOR_VERDE}; color: black;'
                    return df_st

                estilo_tabla_rk = [
                    {'selector': 'th', 'props': [('background-color', COLOR_VERDE), ('color', 'white'), ('font-size', '11px'), ('text-align', 'center')]},
                    {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '11px')]},
                    {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('background-color', '#f4f4f4'), ('color', 'black')]}
                ]

                with out_br1:
                    display(HTML(header_html("🏆 TOP 10: MAYOR CRECIMIENTO")))
                    st_top = df_top10.style.format({'Primas Actuales': fmt_m, 'Crecimiento (%)': fmt_p}, na_rep="-")\
                            .apply(c_rk, subset=['Crecimiento (%)']).apply(b_total, axis=None).set_table_styles(estilo_tabla_rk)
                    display(st_top)

                with out_br2:
                    display(HTML(header_html("📉 TOP 10: MENOR CRECIMIENTO")))
                    st_bot = df_bot10.style.format({'Primas Actuales': fmt_m, 'Crecimiento (%)': fmt_p}, na_rep="-")\
                            .apply(c_rk, subset=['Crecimiento (%)']).apply(b_total, axis=None).set_table_styles(estilo_tabla_rk)
                    display(st_bot)

            # Ensamblar el Grid con 2 columnas completas
            grid = widgets.VBox([
                widgets.HBox([out_hier, out_ml]),
                widgets.HBox([out_mr, out_bl]),
                widgets.HBox([out_br1, out_br2])
            ])
            display(Markdown(f"### 📈 Reporte Analítico YTD (Acumulado a {mes_fin}): {fecha_fin.year} vs {fecha_fin_ant.year}"))
            display(grid)
            display(Markdown("<br><hr><br>"))

        gc.collect()

# 5. Renderizado UI
# ------------------------------------------------------------------------------
ui_tiempo = widgets.VBox([widgets.HBox([dropdown_anio_inicio, dropdown_mes_inicio]), widgets.HBox([dropdown_anio_fin, dropdown_mes_fin, widgets.Label(" | "), input_meses, widgets.Label(" | "), select_anios_atras]), mensaje_alerta], layout={'border': '1px solid #ddd', 'padding': '10px', 'margin': '10px 0'})
ui_datos = widgets.HBox([select_ramos, select_companias, select_regionales])
ui = widgets.VBox([ui_tiempo, ui_datos, widgets.VBox([boton_generar], layout={'align_items': 'center', 'margin': '15px 0'})])
boton_generar.on_click(generar_tablero_maestro)
display(Markdown("---"))
display(ui, out_main)

#Cargue de información

In [ ]:
# 1. CARGA DE DATOS
import pandas as pd
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

df_fc = pd.read_parquet(ruta_fasecolda)
df_ca = pd.read_parquet(ruta_canales)

In [ ]:
df_fc.info()
df_ca.info()

# Tablero anual por Ramo/ año

In [ ]:
# ==============================================================================
# BLOQUE 8.2: TABLERO ANUAL - RAMO (INDUSTRIA, CANALES, BOLIVAR)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS Y LÍMITES TEMPORALES
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_ca = pd.read_parquet(ruta_canales)

    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
    df_ca['FECHA'] = pd.to_datetime(df_ca['FECHA'], errors='coerce')

    fecha_maxima_comun = min(df_fc['FECHA'].max(), df_ca['FECHA'].max())
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted([a for a in df_fc['AÑO'].dropna().unique() if a <= fecha_maxima_comun.year])
ramos_unicos = sorted(list(set(df_fc['RAMOS'].dropna().unique()) | set(df_ca['RAMOS'].dropna().unique())))

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_txt = meses_dict.get(fecha_maxima_comun.month, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_ramo = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
sel_anios_atras_ramo = widgets.SelectMultiple(options=[1, 2, 3, 4, 5], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
dd_mes_ramo = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_ramos_ramo = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '150px'})

btn_ramo = widgets.Button(description='Generar Tablero Ramo', button_style='primary', icon='list', layout={'width': '300px', 'height': '40px'})

ui_ramo = widgets.VBox([
    widgets.HBox([dd_anio_ramo, dd_mes_ramo, sel_anios_atras_ramo]),
    sel_ramos_ramo,
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Límite de datos compartidos detectado: <b>{fecha_maxima_comun.strftime('%B %Y')}</b>. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_ramo
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})

out_ramo = widgets.Output()

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_tablero_ramo(b):
    with out_ramo:
        clear_output(wait=True)

        anio_base = dd_anio_ramo.value
        anios_atras_lista = list(sel_anios_atras_ramo.value)
        ramo_obj = list(sel_ramos_ramo.value)
        mes_str = dd_mes_ramo.value
        mes_num = meses_inversos[mes_str]

        if not ramo_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones (Ramo o Años Atrás)."))
            return

        display(Markdown(f"# 📊 % VAR POR AÑO Y RAMO (Acumulado a {mes_str})"))
        display(Markdown("---"))

        def fmt_m(v): return "-" if pd.isna(v) else f"${v:,.0f}"
        def fmt_p(v): return "-" if pd.isna(v) else f"{v:+.2%}"
        def fmt_v(v): return "-" if pd.isna(v) else f"{v:.2f}x"

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            m_fc = (df_fc['FECHA'].dt.month == mes_num) & (df_fc['RAMOS'].isin(ramo_obj)) & (df_fc['AÑO'].isin([anio_base, anio_ant]))
            m_ca = (df_ca['FECHA'].dt.month <= mes_num) & (df_ca['RAMOS'].isin(ramo_obj)) & (df_ca['AÑO'].isin([anio_base, anio_ant]))

            df_fc_fil = df_fc[m_fc]
            df_ca_fil = df_ca[m_ca]

            if df_fc_fil.empty:
                display(Markdown(f"⚠️ No hay datos en Industria para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'RAMOS'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            bol = df_fc_fil[df_fc_fil['COMPANIA'] == 'BOLIVAR'].groupby(['AÑO', 'RAMOS'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            can = df_ca_fil.groupby(['AÑO', 'RAMOS'])['REAL'].sum().unstack('AÑO').fillna(0)

            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = 0
                if a not in bol.columns: bol[a] = 0
                if a not in can.columns: can[a] = 0

            cols = pd.MultiIndex.from_tuples([
                ('INDUSTRIA', lbl_anterior), ('INDUSTRIA', lbl_actual), ('INDUSTRIA', 'Var'),
                ('CANAL', lbl_anterior), ('CANAL', lbl_actual), ('CANAL', 'Var'), ('CANAL', 'Veces'),
                ('BOLIVAR', lbl_anterior), ('BOLIVAR', lbl_actual), ('BOLIVAR', 'Var'), ('BOLIVAR', 'Veces')
            ])
            res = pd.DataFrame(index=mkt.index, columns=cols)

            res[('INDUSTRIA', lbl_anterior)] = mkt.get(anio_ant, pd.Series(0, index=mkt.index))
            res[('INDUSTRIA', lbl_actual)] = mkt.get(anio_base, pd.Series(0, index=mkt.index))
            res[('INDUSTRIA', 'Var')] = (res[('INDUSTRIA', lbl_actual)] / res[('INDUSTRIA', lbl_anterior)].replace(0, np.nan)) - 1

            res[('CANAL', lbl_anterior)] = can.get(anio_ant, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('CANAL', lbl_actual)] = can.get(anio_base, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('CANAL', 'Var')] = (res[('CANAL', lbl_actual)] / res[('CANAL', lbl_anterior)].replace(0, np.nan)) - 1
            res[('CANAL', 'Veces')] = res[('CANAL', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            res[('BOLIVAR', lbl_anterior)] = bol.get(anio_ant, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('BOLIVAR', lbl_actual)] = bol.get(anio_base, pd.Series(0, index=mkt.index)).reindex(mkt.index).fillna(0)
            res[('BOLIVAR', 'Var')] = (res[('BOLIVAR', lbl_actual)] / res[('BOLIVAR', lbl_anterior)].replace(0, np.nan)) - 1
            res[('BOLIVAR', 'Veces')] = res[('BOLIVAR', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            orden_filas = sorted([r for r in ramo_obj if r in res.index])
            res = res.loc[orden_filas]

            tot_series = res.sum()
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                ant_tot = tot_series.get((ent, lbl_anterior), 0)
                act_tot = tot_series.get((ent, lbl_actual), 0)
                tot_series[(ent, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

            v_m_tot = tot_series.get(('INDUSTRIA', 'Var'), np.nan)
            tot_series[('CANAL', 'Veces')] = (tot_series.get(('CANAL', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            tot_series[('BOLIVAR', 'Veces')] = (tot_series.get(('BOLIVAR', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan

            res.loc['TOTAL'] = tot_series

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))
            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            # ==================================================================
            # CORRECCIÓN DE ESTILOS Y MEMORIA
            # ==================================================================
            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                    if (ent, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(ent, 'Var')], errors='coerce')
                        df_st[(ent, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (ent, 'Veces') in data.columns:
                        v_vec = pd.to_numeric(data[(ent, 'Veces')], errors='coerce')
                        df_st[(ent, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_vec]

                    m_sin = data[(ent, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (ent, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (ent, 'Veces') in data.columns: df_st.loc[m_sin, (ent, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'td:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

        gc.collect()

btn_ramo.on_click(generar_tablero_ramo)
display(ui_ramo, out_ramo)

#Por Sucursal

In [ ]:
# ==============================================================================
# BLOQUE 8.5: TABLERO ANUAL - SUCURSAL (INDUSTRIA, CANALES, BOLIVAR)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS Y LÍMITES TEMPORALES
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_ca = pd.read_parquet(ruta_canales)

    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
    df_ca['FECHA'] = pd.to_datetime(df_ca['FECHA'], errors='coerce')

    fecha_maxima_comun = min(df_fc['FECHA'].max(), df_ca['FECHA'].max())
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted([a for a in df_fc['AÑO'].dropna().unique() if a <= fecha_maxima_comun.year])
ramos_unicos = sorted(list(set(df_fc['RAMOS'].dropna().unique()) | set(df_ca['RAMOS'].dropna().unique())))
sucursales_unicas = sorted(list(set(df_fc['SUCURSAL'].dropna().unique()) | set(df_ca['SUCURSAL'].dropna().unique())))

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_txt = meses_dict.get(fecha_maxima_comun.month, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_suc = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_suc = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_suc = widgets.SelectMultiple(options=[1, 2, 3, 4, 5], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_ramos_suc = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})
sel_sucursales = widgets.SelectMultiple(options=sucursales_unicas, value=[], description='📍 Sucursales:', layout={'width': '350px', 'height': '120px'})

btn_suc = widgets.Button(description='Generar Tablero Sucursal', button_style='primary', icon='sitemap', layout={'width': '300px', 'height': '40px'})

ui_suc = widgets.VBox([
    widgets.HBox([dd_anio_suc, dd_mes_suc, sel_anios_atras_suc]),
    widgets.HBox([sel_sucursales, sel_ramos_suc]),
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Límite de datos compartidos detectado: <b>{fecha_maxima_comun.strftime('%B %Y')}</b>. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_suc
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})

out_suc = widgets.Output()

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_tablero_sucursal(b):
    with out_suc:
        clear_output(wait=True)

        anio_base = dd_anio_suc.value
        anios_atras_lista = list(sel_anios_atras_suc.value)
        ramo_obj = list(sel_ramos_suc.value)
        suc_obj = list(sel_sucursales.value)
        mes_str = dd_mes_suc.value
        mes_num = meses_inversos[mes_str]

        if not ramo_obj or not suc_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones (Ramo, Sucursal o Años Atrás)."))
            return

        display(Markdown(f"# 📊 % VAR POR AÑO Y SUCURSAL (Acumulado a {mes_str})"))
        display(Markdown("---"))

        def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
        def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
        def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            m_fc = (df_fc['FECHA'].dt.month == mes_num) & (df_fc['RAMOS'].isin(ramo_obj)) & (df_fc['SUCURSAL'].isin(suc_obj)) & (df_fc['AÑO'].isin([anio_base, anio_ant]))
            m_ca = (df_ca['FECHA'].dt.month <= mes_num) & (df_ca['RAMOS'].isin(ramo_obj)) & (df_ca['SUCURSAL'].isin(suc_obj)) & (df_ca['AÑO'].isin([anio_base, anio_ant]))

            df_fc_fil = df_fc[m_fc]
            df_ca_fil = df_ca[m_ca]

            if df_fc_fil.empty and df_ca_fil.empty:
                display(Markdown(f"⚠️ No hay datos en ninguna base para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'SUCURSAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            bol = df_fc_fil[df_fc_fil['COMPANIA'] == 'BOLIVAR'].groupby(['AÑO', 'SUCURSAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            can = df_ca_fil.groupby(['AÑO', 'SUCURSAL'])['REAL'].sum().unstack('AÑO').fillna(0)

            todos_indices = sorted(list(set(mkt.index) | set(bol.index) | set(can.index)))

            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = pd.Series(0, index=mkt.index)
                if a not in bol.columns: bol[a] = pd.Series(0, index=bol.index)
                if a not in can.columns: can[a] = pd.Series(0, index=can.index)

            cols = pd.MultiIndex.from_tuples([
                ('INDUSTRIA', lbl_anterior), ('INDUSTRIA', lbl_actual), ('INDUSTRIA', 'Var'),
                ('CANAL', lbl_anterior), ('CANAL', lbl_actual), ('CANAL', 'Var'), ('CANAL', 'Veces'),
                ('BOLIVAR', lbl_anterior), ('BOLIVAR', lbl_actual), ('BOLIVAR', 'Var'), ('BOLIVAR', 'Veces')
            ])
            res = pd.DataFrame(index=todos_indices, columns=cols)

            # Llenado
            res[('INDUSTRIA', lbl_anterior)] = mkt[anio_ant].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', lbl_actual)] = mkt[anio_base].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', 'Var')] = (res[('INDUSTRIA', lbl_actual)] / res[('INDUSTRIA', lbl_anterior)].replace(0, np.nan)) - 1

            res[('CANAL', lbl_anterior)] = can[anio_ant].reindex(todos_indices).fillna(0)
            res[('CANAL', lbl_actual)] = can[anio_base].reindex(todos_indices).fillna(0)
            res[('CANAL', 'Var')] = (res[('CANAL', lbl_actual)] / res[('CANAL', lbl_anterior)].replace(0, np.nan)) - 1
            res[('CANAL', 'Veces')] = res[('CANAL', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            res[('BOLIVAR', lbl_anterior)] = bol[anio_ant].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', lbl_actual)] = bol[anio_base].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', 'Var')] = (res[('BOLIVAR', lbl_actual)] / res[('BOLIVAR', lbl_anterior)].replace(0, np.nan)) - 1
            res[('BOLIVAR', 'Veces')] = res[('BOLIVAR', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            orden_filas = sorted([r for r in suc_obj if r in res.index])
            res = res.loc[orden_filas]

            # Cálculo de la fila TOTAL
            tot_series = res.sum()
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                ant_tot = tot_series.get((ent, lbl_anterior), 0)
                act_tot = tot_series.get((ent, lbl_actual), 0)
                tot_series[(ent, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

            v_m_tot = tot_series.get(('INDUSTRIA', 'Var'), np.nan)
            tot_series[('CANAL', 'Veces')] = (tot_series.get(('CANAL', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            tot_series[('BOLIVAR', 'Veces')] = (tot_series.get(('BOLIVAR', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            res.loc['TOTAL'] = tot_series

            # --- APLICAR TEXTO "Sin registros" ---
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                mask = (res[(ent, lbl_anterior)] == 0) & (res[(ent, lbl_actual)] == 0) & (res.index != 'TOTAL')
                if mask.any():
                    cols_to_cast = [(ent, lbl_anterior), (ent, lbl_actual), (ent, 'Var')]
                    if (ent, 'Veces') in res.columns: cols_to_cast.append((ent, 'Veces'))
                    for col in cols_to_cast:
                        res[col] = res[col].astype(object)
                    res.loc[mask, [(ent, lbl_anterior), (ent, lbl_actual)]] = "Sin registros"
                    res.loc[mask, (ent, 'Var')] = "-"
                    if (ent, 'Veces') in res.columns: res.loc[mask, (ent, 'Veces')] = "-"

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))
            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            def estilo_celdas(data):
                # Usamos dtype=object
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                    if (ent, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(ent, 'Var')], errors='coerce')
                        # Listas de comprensión
                        df_st[(ent, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (ent, 'Veces') in data.columns:
                        v_vec = pd.to_numeric(data[(ent, 'Veces')], errors='coerce')
                        # Listas de comprensión
                        df_st[(ent, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_vec]

                    m_sin = data[(ent, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (ent, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (ent, 'Veces') in data.columns: df_st.loc[m_sin, (ent, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    # Forzar el texto en el index TOTAL
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'td:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

        gc.collect()

btn_suc.on_click(generar_tablero_sucursal)
display(ui_suc, out_suc)

#Por Regional

In [ ]:
# ==============================================================================
# BLOQUE 8.3: TABLERO ANUAL - REGIONAL (INDUSTRIA, CANALES, BOLIVAR)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
import gc
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARGA DE DATOS Y LÍMITES TEMPORALES
# ------------------------------------------------------------------------------
ruta_fasecolda = 'Base_Maestra_Acumulada.parquet'
ruta_canales = 'Primas_Canales_Acumulada.parquet'

try:
    df_fc = pd.read_parquet(ruta_fasecolda)
    df_ca = pd.read_parquet(ruta_canales)

    df_fc['FECHA'] = pd.to_datetime(df_fc['FECHA'], errors='coerce')
    df_ca['FECHA'] = pd.to_datetime(df_ca['FECHA'], errors='coerce')

    fecha_maxima_comun = min(df_fc['FECHA'].max(), df_ca['FECHA'].max())
except Exception as e:
    display(Markdown(f"⚠️ **Error al cargar los archivos:** {e}"))

# ------------------------------------------------------------------------------
# 2. PARÁMETROS PARA WIDGETS
# ------------------------------------------------------------------------------
anios_disponibles = sorted([a for a in df_fc['AÑO'].dropna().unique() if a <= fecha_maxima_comun.year])
ramos_unicos = sorted(list(set(df_fc['RAMOS'].dropna().unique()) | set(df_ca['RAMOS'].dropna().unique())))
regionales_unicas = sorted(list(set(df_fc['REGIONAL'].dropna().unique()) | set(df_ca['REGIONAL'].dropna().unique())))

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}

ultimo_anio = anios_disponibles[-1] if anios_disponibles else 2024
ultimo_mes_txt = meses_dict.get(fecha_maxima_comun.month, 'Diciembre')

# ------------------------------------------------------------------------------
# 3. INTERFAZ (UI)
# ------------------------------------------------------------------------------
dd_anio_reg = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
dd_mes_reg = widgets.Dropdown(options=list(meses_dict.values()), value=ultimo_mes_txt, description='🗓️ Mes (YTD):', layout={'width': '200px'})
sel_anios_atras_reg = widgets.SelectMultiple(options=[1, 2, 3, 4, 5], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})

sel_ramos_reg = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '120px'})
sel_reg_reg = widgets.SelectMultiple(options=regionales_unicas, value=[], description='🗺️ Regionales:', layout={'width': '350px', 'height': '120px'})

btn_reg = widgets.Button(description='Generar Tablero Regional', button_style='primary', icon='map', layout={'width': '300px', 'height': '40px'})

ui_reg = widgets.VBox([
    widgets.HBox([dd_anio_reg, dd_mes_reg, sel_anios_atras_reg]),
    widgets.HBox([sel_reg_reg, sel_ramos_reg]),
    widgets.HTML(f"<em style='color:gray; font-size:11px;'>* Límite de datos compartidos detectado: <b>{fecha_maxima_comun.strftime('%B %Y')}</b>. Usa Ctrl/Cmd para selección múltiple.</em>"),
    btn_reg
], layout={'border': '1px solid #ddd', 'padding': '10px', 'background-color': '#fcfcfc'})

out_reg = widgets.Output()

# ------------------------------------------------------------------------------
# 4. MOTOR ANALÍTICO
# ------------------------------------------------------------------------------
def generar_tablero_regional(b):
    with out_reg:
        clear_output(wait=True)

        anio_base = dd_anio_reg.value
        anios_atras_lista = list(sel_anios_atras_reg.value)
        ramo_obj = list(sel_ramos_reg.value)
        reg_obj = list(sel_reg_reg.value)
        mes_str = dd_mes_reg.value
        mes_num = meses_inversos[mes_str]

        if not ramo_obj or not reg_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones (Ramo, Regional o Años Atrás)."))
            return

        display(Markdown(f"# 📊 % VAR POR AÑO Y REGIONAL (Acumulado a {mes_str})"))
        display(Markdown("---"))

        def fmt_m(v): return v if isinstance(v, str) else ("-" if pd.isna(v) or v == 0 else f"${v:,.0f}")
        def fmt_p(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:+.2%}")
        def fmt_v(v): return v if isinstance(v, str) else ("-" if pd.isna(v) else f"{v:.2f}x")

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"

            m_fc = (df_fc['FECHA'].dt.month == mes_num) & (df_fc['RAMOS'].isin(ramo_obj)) & (df_fc['REGIONAL'].isin(reg_obj)) & (df_fc['AÑO'].isin([anio_base, anio_ant]))
            m_ca = (df_ca['FECHA'].dt.month <= mes_num) & (df_ca['RAMOS'].isin(ramo_obj)) & (df_ca['REGIONAL'].isin(reg_obj)) & (df_ca['AÑO'].isin([anio_base, anio_ant]))

            df_fc_fil = df_fc[m_fc]
            df_ca_fil = df_ca[m_ca]

            if df_fc_fil.empty and df_ca_fil.empty:
                display(Markdown(f"⚠️ No hay datos en ninguna base para la comparativa **{anio_ant} vs {anio_base}** en el mes de {mes_str}."))
                continue

            mkt = df_fc_fil.groupby(['AÑO', 'REGIONAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            bol = df_fc_fil[df_fc_fil['COMPANIA'] == 'BOLIVAR'].groupby(['AÑO', 'REGIONAL'])['TOTAL'].sum().unstack('AÑO').fillna(0)
            can = df_ca_fil.groupby(['AÑO', 'REGIONAL'])['REAL'].sum().unstack('AÑO').fillna(0)

            # Unión inteligente de índices para que "Gerencia" o similares no se pierdan
            todos_indices = sorted(list(set(mkt.index) | set(bol.index) | set(can.index)))

            for a in [anio_ant, anio_base]:
                if a not in mkt.columns: mkt[a] = pd.Series(0, index=mkt.index)
                if a not in bol.columns: bol[a] = pd.Series(0, index=bol.index)
                if a not in can.columns: can[a] = pd.Series(0, index=can.index)

            cols = pd.MultiIndex.from_tuples([
                ('INDUSTRIA', lbl_anterior), ('INDUSTRIA', lbl_actual), ('INDUSTRIA', 'Var'),
                ('CANAL', lbl_anterior), ('CANAL', lbl_actual), ('CANAL', 'Var'), ('CANAL', 'Veces'),
                ('BOLIVAR', lbl_anterior), ('BOLIVAR', lbl_actual), ('BOLIVAR', 'Var'), ('BOLIVAR', 'Veces')
            ])
            res = pd.DataFrame(index=todos_indices, columns=cols)

            # Llenado
            res[('INDUSTRIA', lbl_anterior)] = mkt[anio_ant].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', lbl_actual)] = mkt[anio_base].reindex(todos_indices).fillna(0)
            res[('INDUSTRIA', 'Var')] = (res[('INDUSTRIA', lbl_actual)] / res[('INDUSTRIA', lbl_anterior)].replace(0, np.nan)) - 1

            res[('CANAL', lbl_anterior)] = can[anio_ant].reindex(todos_indices).fillna(0)
            res[('CANAL', lbl_actual)] = can[anio_base].reindex(todos_indices).fillna(0)
            res[('CANAL', 'Var')] = (res[('CANAL', lbl_actual)] / res[('CANAL', lbl_anterior)].replace(0, np.nan)) - 1
            res[('CANAL', 'Veces')] = res[('CANAL', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            res[('BOLIVAR', lbl_anterior)] = bol[anio_ant].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', lbl_actual)] = bol[anio_base].reindex(todos_indices).fillna(0)
            res[('BOLIVAR', 'Var')] = (res[('BOLIVAR', lbl_actual)] / res[('BOLIVAR', lbl_anterior)].replace(0, np.nan)) - 1
            res[('BOLIVAR', 'Veces')] = res[('BOLIVAR', 'Var')] / res[('INDUSTRIA', 'Var')].replace(0, np.nan)

            orden_filas = sorted([r for r in reg_obj if r in res.index])
            res = res.loc[orden_filas]

            # Cálculo de la fila TOTAL
            tot_series = res.sum()
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                ant_tot = tot_series.get((ent, lbl_anterior), 0)
                act_tot = tot_series.get((ent, lbl_actual), 0)
                tot_series[(ent, 'Var')] = (act_tot / ant_tot) - 1 if ant_tot != 0 else np.nan

            v_m_tot = tot_series.get(('INDUSTRIA', 'Var'), np.nan)
            tot_series[('CANAL', 'Veces')] = (tot_series.get(('CANAL', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            tot_series[('BOLIVAR', 'Veces')] = (tot_series.get(('BOLIVAR', 'Var'), np.nan) / v_m_tot) if pd.notna(v_m_tot) and v_m_tot != 0 else np.nan
            res.loc['TOTAL'] = tot_series

            # --- APLICAR TEXTO "Sin registros" ---
            for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                mask = (res[(ent, lbl_anterior)] == 0) & (res[(ent, lbl_actual)] == 0) & (res.index != 'TOTAL')
                if mask.any():
                    cols_to_cast = [(ent, lbl_anterior), (ent, lbl_actual), (ent, 'Var')]
                    if (ent, 'Veces') in res.columns: cols_to_cast.append((ent, 'Veces'))
                    for col in cols_to_cast:
                        res[col] = res[col].astype(object)
                    res.loc[mask, [(ent, lbl_anterior), (ent, lbl_actual)]] = "Sin registros"
                    res.loc[mask, (ent, 'Var')] = "-"
                    if (ent, 'Veces') in res.columns: res.loc[mask, (ent, 'Veces')] = "-"

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))
            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in res.columns}

            # ==================================================================
            # CORRECCIÓN DE ESTILOS Y MEMORIA
            # ==================================================================
            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns, dtype=object)
                for ent in ['INDUSTRIA', 'CANAL', 'BOLIVAR']:
                    if (ent, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(ent, 'Var')], errors='coerce')
                        df_st[(ent, 'Var')] = ['color: #1E8449;' if v > 0 else 'color: #E74C3C;' if v < 0 else 'color: #7F8C8D;' for v in v_cia]

                    if (ent, 'Veces') in data.columns:
                        v_vec = pd.to_numeric(data[(ent, 'Veces')], errors='coerce')
                        df_st[(ent, 'Veces')] = ['color: #1E8449; font-weight: bold;' if v > 1.8 else
                                                'color: #F39C12; font-weight: bold;' if v >= 1.0 else
                                                'color: #E74C3C; font-weight: bold;' if pd.notna(v) else 'color: #7F8C8D;' for v in v_vec]

                    m_sin = data[(ent, lbl_actual)] == "Sin registros"
                    if m_sin.any():
                        df_st.loc[m_sin, (ent, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                        df_st.loc[m_sin, (ent, 'Var')] = 'text-align: center; color: #bdc3c7;'
                        if (ent, 'Veces') in data.columns: df_st.loc[m_sin, (ent, 'Veces')] = 'text-align: center; color: #bdc3c7;'

                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = df_st.loc['TOTAL'].astype(str) + ' background-color: #d1f2eb; font-weight: bold; border-top: 2px solid #0F753B;'
                return df_st

            display(res.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('border-right', '1px solid white')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '11px')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '12px')]},
                {'selector': 'td:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'td:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(4)', 'props': [('border-right', '2px solid #bdc3c7')]},
                {'selector': 'th.col_heading.level1:nth-child(8)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

        gc.collect()

btn_reg.on_click(generar_tablero_regional)
display(ui_reg, out_reg)